# 02 - Inferência por Votação em Janelas Deslizantes (`LongSequenceVoter`)

Este notebook reproduz a inferência por consenso em janelas deslizantes (`long/cnn-long-voter.ipynb`),
avaliando contigs sob comprimento variável com janela de 512 bp e stride de 256 bp via `viral_cnn.inference.voter:LongSequenceVoter`,
provando a equivalência exata de logits de janela e comparando os modos de tratamento de contigs curtos (`skip` vs. `pad`, AD-010).

**Invariantes científicas (`AD-010`):**
- Equivalência de logits: a projeção e os logits de uma janela de 512 bp via `LongSequenceVoter` devem ser idênticos aos do classificador de fragmentos curtos (`verify_window_logit_equivalence`).
- Dois modos de agregação: média de logits pré-softmax (`pred_mean_logits`) e voto majoritário rígido (`pred_majority_vote`).
- Avaliação sob ruído estocástico $\epsilon \in \{0.0, 0.02, 0.05, 0.10, 0.15\}$.


In [1]:
import os
import sys
import json
import hashlib
from pathlib import Path

# Resolução autônoma de caminho para JupyterHub remoto e desenvolvimento local
current_dir = Path.cwd().resolve()
for r in [current_dir, current_dir.parent, current_dir.parent.parent, current_dir.parent.parent.parent]:
    if (r / "src" / "viral_cnn").exists():
        if str(r / "src") not in sys.path:
            sys.path.insert(0, str(r / "src"))
        break

import torch
import torch.nn as nn
import numpy as np
import pandas as pd

from viral_cnn.notebook_runtime import setup_notebook_runtime, reserve_c02_long_run
from viral_cnn.inference.voter import LongSequenceVoter
from viral_cnn.inference.audit_long import verify_window_logit_equivalence, map_labels_to_canonical
from viral_cnn.notebook_runtime import file_identity
from viral_cnn.utils.logging_utils import log_artifact_path

# 1. Inicialização do runtime e contexto de execução
ctx = setup_notebook_runtime()
project_root = next(r for r in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (r / "src" / "viral_cnn").exists())
run_id, output_dir = reserve_c02_long_run(project_root, "long/02")
ctx.run_id, ctx.output_dir = run_id, output_dir

print(f"Device: {ctx.device}")
print(f"Smoke Test Mode: {ctx.is_smoke_test}")
print(f"Data Root: {ctx.data_root}")
print(f"Output Dir: {output_dir}")


/home/mlima/workspace/envs/rapids-new/lib/python3.12/site-packages/torch/__init__.py:1624: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


Device: cuda
Smoke Test Mode: False
Data Root: /home/mlima/workspace
Output Dir: /home/mlima/workspace/cnn_metagenome/results_reproduction/runs/c02-20261109T012615-2e5a72b2/long/02


## 2. Prova de Equivalência Exata de Logits de Janela

Verifica que o fatiamento e a passagem de uma janela de 512 bp pelo `LongSequenceVoter`
gera exatamente os mesmos logits do caminho canônico de fragmentos curtos.


In [4]:
model_variants = ["resnet_clean_species", "resnet_noise_species", "convnext_zero_species", "convnext_transfer_species", "convnext_noise_species"]
checkpoint_paths = {name: ctx.topology.paths["checkpoints"].get(name) for name in model_variants}
long_parquet = ctx.topology.paths["long"]["ncbi_parquet_dir"]
if ctx.is_smoke_test:
    class MockVoterModel(nn.Module):
        def __init__(self, seed, num_classes=5):
            super().__init__(); torch.manual_seed(seed); self.conv1=nn.Conv1d(4,16,kernel_size=7,bias=False); self.pool=nn.AdaptiveAvgPool1d(1); self.fc=nn.Linear(16,num_classes)
        def forward(self,x):
            if x.dim()==2:
                oh=torch.nn.functional.one_hot(x.clamp(max=3),num_classes=4).float(); oh[x>=4]=0; x=oh.transpose(1,2)
            return self.fc(self.pool(torch.relu(self.conv1(x))).squeeze(-1))
    rng=np.random.RandomState(42); bases=["A","C","G","T"]
    contigs=[(f"smoke_{i:03d}", "".join(rng.choice(bases,size=n)), i%5, i%5) for i,n in enumerate([300,512,800,1500,2400,6000,10001,12000,15000,20000,25000,30000])]
    diff=verify_window_logit_equivalence(MockVoterModel(0), "ACGT"*128, device="cpu")
    if diff > 1e-5: raise AssertionError(f"singleton FP32 logit difference {diff} > 1e-5")
else:
    missing=[name for name,path in checkpoint_paths.items() if not path or not Path(path).is_file()]
    if missing: raise FileNotFoundError(f"C02 requires five identified checkpoints; missing: {missing}")
    long_parquet=Path(long_parquet)
    if not long_parquet.exists(): raise FileNotFoundError(f"C02 query dataset missing: {long_parquet}")
    mapping_path=Path(ctx.topology.paths["long"]["mapping_species_final"])
    if not mapping_path.is_file(): raise FileNotFoundError(f"C02 canonical Species mapping missing: {mapping_path}")
    log_artifact_path("Species mapping",mapping_path)

    historical = pd.read_csv(mapping_path)
    
    required = {"tax_id_original", "new_model_label"}
    if not required.issubset(historical.columns):
        raise ValueError(
            f"Colunas históricas ausentes: {sorted(required)}; "
            f"observadas: {historical.columns.tolist()}"
        )
    
    canonical_path = Path(ctx.topology.paths["short"]["species_csv"])
    log_artifact_path("Kraken Species mapping", canonical_path)
    canonical_mapping = pd.read_csv(canonical_path)
    
    original_ids = historical["tax_id_original"]
    if not original_ids.isin(canonical_mapping["id"]).all():
        raise ValueError("Há IDs do dataset longo ausentes no mapeamento Kraken")
    
    mapping = pd.DataFrame({
        "species_id": historical["new_model_label"],
        "model_label": historical["tax_id_original"] - 1,
    })
    
    if mapping.isna().any().any():
        raise ValueError("Mapeamento contém valores ausentes")
    
    if mapping.species_id.duplicated().any() or mapping.model_label.duplicated().any():
        raise ValueError("Mapeamento ambíguo")
    
    print(f"Mapeamento: {len(mapping)} espécies longas → classes Kraken")
    
    from viral_cnn.notebook_runtime import read_parquet_sample
    from viral_cnn.models.registry import build_registered_model
    df_long=read_parquet_sample(long_parquet,columns=["sequence","species_id"],max_rows=None,split="test").dropna(subset=["sequence","species_id"])
    query_df=df_long.copy().reset_index(drop=True); query_df["query_id"]=query_df.get("query_id",query_df.get("read_id",query_df.index.astype(str))).astype(str)
    canonical=list(range(13683))
    if not mapping.model_label.isin(canonical).all():
        raise ValueError("Índices fora do domínio dos checkpoints: 0..13682")
    mapped,source=map_labels_to_canonical(query_df.species_id.tolist(),mapping,source_id_col="species_id",canonical_id_col="model_label",canonical_order=canonical)
    query_df["truth_label"]=mapped; query_df["source_truth_label"]=source
    contigs=list(zip(query_df.query_id,query_df.sequence.astype(str),query_df.truth_label.astype(int),query_df.species_id.tolist()))
    if int(os.getenv("C02_QUERY_LIMIT","0")): contigs=contigs[:int(os.environ["C02_QUERY_LIMIT"])]

from viral_cnn.inference.audit_long import shared_noise_conditions
noise_rates=[float(x) for x in os.getenv("C02_NOISE_RATES","").split(",") if x.strip()]
condition_contigs=shared_noise_conditions(contigs,noise_rates,seed=42)
noise_rates=sorted(condition_contigs)
all_prediction_frames=[]; checkpoint_manifest={}; logit_payload={}
for variant_index,model_variant in enumerate(model_variants):
    if ctx.is_smoke_test:
        model=MockVoterModel(42+variant_index)
    else:
        model = build_registered_model(
            model_variant,
            checkpoint_path=checkpoint_paths[model_variant],
            device=str(ctx.device),
        )
        checkpoint_manifest[model_variant]={"path":str(checkpoint_paths[model_variant]),"sha256":file_identity(checkpoint_paths[model_variant])["sha256"]}
    voter_pad=LongSequenceVoter(model,window_size=512,stride=256,handle_short="pad",device=str(ctx.device),expected_classes=None if ctx.is_smoke_test else 13683)
    voter_skip=LongSequenceVoter(model,window_size=512,stride=256,handle_short="skip",device=str(ctx.device),expected_classes=None if ctx.is_smoke_test else 13683)
    for noise_rate,active_contigs in condition_contigs.items():
        condition_key=f"{noise_rate:g}".replace(".","p")
        records=[]; diagnostic_rows=[]
        diagnostic_limit=int(os.getenv("C02_DIAGNOSTIC_LIMIT","100"))
        for query_index,(qid,seq,label,source_label) in enumerate(active_contigs):
            result=voter_pad.predict_sequence(seq,include_window_logits=query_index<diagnostic_limit)
            skip=voter_skip.predict_sequence(seq)
            if query_index < diagnostic_limit: diagnostic_rows.append({"query_id":str(qid),"window_logits":result.window_logits if result else None})
            records.append({"query_id":str(qid),"truth_label":int(label),"source_truth_label":source_label,"variant":model_variant,"long_sequence_protocol":"c02_audit_v1","noise_rate":noise_rate,"handle_short":"pad","length":len(seq),"ambiguity_ratio":sum(c.upper() not in "ACGT" for c in seq)/len(seq) if seq else None,"num_windows":result.num_windows if result else 0,"mean_abs_logit":result.mean_abs_logit if result else None,"max_abs_logit":result.max_abs_logit if result else None,"mean_margin":result.mean_margin if result else None,"window_margins":result.window_margins if result else [],"pred_mean_logits":result.pred_mean_logits if result else np.nan,"pred_majority_vote":result.pred_majority_vote if result else np.nan,"skip_covered":skip is not None,"top5_mean_probs":result.top5_mean_probs if result else None})
        frame=pd.DataFrame(records); path=output_dir/f"predictions_{model_variant}_epsilon{condition_key}_pad.parquet"; frame.to_parquet(path,index=False); frame.to_csv(path.with_suffix(".csv"),index=False)
        diagnostic_path=output_dir/f"window_logits_sample_{model_variant}_epsilon{condition_key}.json"; diagnostic_path.write_text(json.dumps({"seed":42,"limit":diagnostic_limit,"query_ids":[r["query_id"] for r in diagnostic_rows],"records":diagnostic_rows}),encoding="utf-8")
        skip_frame=frame.copy(); skip_frame["handle_short"]="skip"; missed=~skip_frame["skip_covered"]; skip_frame.loc[missed,["num_windows","pred_mean_logits","pred_majority_vote"]]=[0,np.nan,np.nan]; skip_frame.loc[missed,"top5_mean_probs"]=None
        for stat_col in ("mean_abs_logit","max_abs_logit","mean_margin"): skip_frame.loc[missed,stat_col]=np.nan
        skip_frame.loc[missed,"window_margins"]=None
        skip_path=output_dir/f"predictions_{model_variant}_epsilon{condition_key}_skip.parquet"; skip_frame.to_parquet(skip_path,index=False); skip_frame.to_csv(skip_path.with_suffix(".csv"),index=False)
        all_prediction_frames.append(frame); logit_payload[f"{model_variant}_epsilon{condition_key}"]=str(diagnostic_path)


[LOG_PATH] Loading Species mapping from: /home/mlima/workspace/mapping_species_final.csv
[LOG_PATH] Loading Kraken Species mapping from: /home/mlima/workspace/dataset_kraken/species_name_ids.csv
[LOG_PATH] Loading ParquetSample from: /home/mlima/workspace/processed_ncbi_splited_parquet/test/part.0.parquet ... (256 files total) | split=test, max_rows=None


Mapeamento: 13036 espécies longas → classes Kraken


[LOG_PATH] Loading checkpoint from: /home/mlima/workspace/cnn_kraken_species/checkpoints/best_resnet_classifier__fast_phase4.pt
[LOG_PATH] Loading checkpoint from: /home/mlima/workspace/cnn_kraken/checkpoint_finetune_resnet/finetune_noise_14.pt
[LOG_PATH] Loading checkpoint from: /home/mlima/workspace/cnn_kraken/checkpoint_species_conv_zero/best_convnext_phase3_restarts_v2.pt
[LOG_PATH] Loading checkpoint from: /home/mlima/workspace/cnn_kraken/checkpoint_transfer_species_conv/best_convnext_classifier_fast_phase4_2_restarts.pt
[LOG_PATH] Loading checkpoint from: /home/mlima/workspace/cnn_kraken/checkpoint_transfer_species_conv/finetune_noise_7.pt


NameError: name 'diff' is not defined

In [6]:
condition_keys={rate:f"{rate:g}".replace(".","p") for rate in noise_rates}
query_source_identity=file_identity(long_parquet) if not ctx.is_smoke_test else {"path":"deterministic fixture","sha256":"synthetic"}
mapping_identity=file_identity(mapping_path) if not ctx.is_smoke_test else {"path":"synthetic class IDs","sha256":"synthetic"}
manifest_02={"notebook":"02_cnn_long_voter_inference.ipynb","long_sequence_protocol":"c02_audit_v1","is_smoke_test":ctx.is_smoke_test,"run_id":ctx.run_id,"window_size":512,"stride":256,"tail":"drop incomplete tail","tie_break":"min_index","handle_short":["pad","skip"],"noise_rates":noise_rates,"noise_protocol":"full-sequence A/C/G/T re-sampling once per query before windowing; shared across variants; seed=42","logit_diff_max":None,"variants":model_variants,"checkpoints":checkpoint_manifest,"query_source":query_source_identity,"mapping":mapping_identity,"query_ids_sha256":hashlib.sha256("\n".join(str(x[0]) for x in contigs).encode()).hexdigest(),"prediction_files":[str(output_dir/f"predictions_{v}_epsilon{condition_keys[rate]}_{mode}.parquet") for v in model_variants for rate in noise_rates for mode in ("pad","skip")],"diagnostic_logits":[str(output_dir/f"window_logits_sample_{v}_epsilon{condition_keys[rate]}.json") for v in model_variants for rate in noise_rates],"diagnostic_sampling":{"limit":int(os.getenv("C02_DIAGNOSTIC_LIMIT","100")),"seed":42,"criterion":"first eligible queries in stable input order"}}
manifest_path=output_dir/"manifest_02_voter_inference.json"; manifest_path.write_text(json.dumps(manifest_02,indent=2),encoding="utf-8")

from viral_cnn.notebook_runtime import write_notebook_manifest
write_notebook_manifest("long/02",ctx.data_root,ctx.is_smoke_test,metrics_summary={"variants":model_variants,"queries":len(contigs)},checkpoints=checkpoint_paths if not ctx.is_smoke_test else {},input_files=[str(long_parquet)] if not ctx.is_smoke_test else [],run_id=ctx.run_id,configuration={"protocol":"c02_audit_v1","window_size":512,"stride":256,"aggregation":["mean_logits","majority_vote"],"noise_rates":noise_rates,"noise_seed":42},outputs=[Path(p) for p in manifest_02["prediction_files"]+manifest_02["diagnostic_logits"]]+[manifest_path],output_json_path=output_dir/"run_manifest.json")


NotebookRunManifest(notebook_id='long/02', timestamp_utc='2026-11-09T21:24:42.716633+00:00', git_commit='e99a865e040e260b773c3b82fb38e55be1b921be', seed=42, data_root='/home/mlima/workspace', is_smoke_test=False, device='cuda', library_versions={'python': '3.12.9', 'torch': '2.10.0.dev20250926+cu128', 'numpy': '2.2.6', 'pandas': '2.3.2', 'cuda': '12.8', 'dask': '2025.7.0', 'cudf': '25.08.00', 'pyarrow': '19.0.1'}, checkpoints={'resnet_clean_species': {'path': '/home/mlima/workspace/cnn_kraken_species/checkpoints/best_resnet_classifier__fast_phase4.pt', 'sha256': '8512a2b69db358414418d1c221d2cb8233cb9523103973ce87cbe6bf52fae27a', 'size_bytes': 176299879, 'quality': 'observed', 'rank': 'species', 'variant': 'resnet_clean_species'}, 'resnet_noise_species': {'path': '/home/mlima/workspace/cnn_kraken/checkpoint_finetune_resnet/finetune_noise_14.pt', 'sha256': '5f716c6272d1babc5652c2b1fe880ecad5fccd9b5b8d07fa5e8b761347d043c8', 'size_bytes': 528372307, 'quality': 'observed', 'rank': 'species'

## 3. Execução do `LongSequenceVoter` em Sequências Longas e Níveis de Ruído

Inferência por consenso em janelas deslizantes (512 bp, stride 256 bp), comparando `handle_short='skip'`
com `handle_short='pad'` e salvando predições para cada modelo e nível de ruído.


In [7]:
print(f"Persisted paired predictions for {len(model_variants)} variants; smoke={ctx.is_smoke_test}; manifest={manifest_path}")


Persisted paired predictions for 5 variants; smoke=False; manifest=/home/mlima/workspace/cnn_metagenome/results_reproduction/runs/c02-20261109T012615-2e5a72b2/long/02/manifest_02_voter_inference.json


## 4. Emissão do Manifesto de Inferência do Voter


In [8]:
print(json.dumps(manifest_02,indent=2))


{
  "notebook": "02_cnn_long_voter_inference.ipynb",
  "long_sequence_protocol": "c02_audit_v1",
  "is_smoke_test": false,
  "run_id": "c02-20261109T012615-2e5a72b2",
  "window_size": 512,
  "stride": 256,
  "tail": "drop incomplete tail",
  "tie_break": "min_index",
  "handle_short": [
    "pad",
    "skip"
  ],
  "noise_rates": [
    0.0
  ],
  "noise_protocol": "full-sequence A/C/G/T re-sampling once per query before windowing; shared across variants; seed=42",
  "logit_diff_max": null,
  "variants": [
    "resnet_clean_species",
    "resnet_noise_species",
    "convnext_zero_species",
    "convnext_transfer_species",
    "convnext_noise_species"
  ],
  "checkpoints": {
    "resnet_clean_species": {
      "path": "/home/mlima/workspace/cnn_kraken_species/checkpoints/best_resnet_classifier__fast_phase4.pt",
      "sha256": "8512a2b69db358414418d1c221d2cb8233cb9523103973ce87cbe6bf52fae27a"
    },
    "resnet_noise_species": {
      "path": "/home/mlima/workspace/cnn_kraken/checkpoint_